# RLHF Assignment

The notebook includes:
- two base models (`distilgpt2`, `gpt2`)
- baseline SFT
- two preference datasets (`Anthropic/hh-rlhf`, `nvidia/HelpSteer2`)
- reward model training
- DPO training
- held-out preference accuracy
- qualitative comparison
- blind human evaluation template

In [1]:
# Install the libraries used throughout the notebook.
!pip -q install transformers datasets accelerate trl peft sentencepiece evaluate scikit-learn rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [2]:
# Import all required libraries.
# - transformers / trl for language-model fine-tuning and DPO
# - datasets for loading public preference datasets
# - evaluate for BLEU/ROUGE/accuracy metrics
import os
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import evaluate

from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from trl import SFTTrainer, DPOTrainer
try:
    from trl import DPOConfig
    HAS_DPOCONFIG = True
except Exception:
    HAS_DPOCONFIG = False

In [3]:
# Global configuration.
# This cell sets random seeds for reproducibility, selects GPU if available,
# creates an output folder, and defines the main experiment sizes/hyperparameters.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

OUTPUT_ROOT = "./assignment_outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

BASE_MODELS = ["distilgpt2", "gpt2"]

SFT_EPOCHS = 5
REWARD_EPOCHS = 1
DPO_EPOCHS = 1

HH_RLHF_TRAIN_PAIRS = 4000
HH_RLHF_HELDOUT_PAIRS = 300

HS_TRAIN_PAIRS = 4000
HS_HELDOUT_PAIRS = 300

MAX_RM_LENGTH = 256
MAX_GEN_INPUT_LENGTH = 512
MAX_CAUSAL_LENGTH = 1024

Device: cuda


## Helper functions

In [4]:
# Utility helpers.
# These functions standardize file names and load models/tokenizers.
# We keep separate loaders for training and evaluation so the code is clearer.
def slugify(text: str) -> str:
    return text.lower().replace("/", "_").replace("-", "_").replace(" ", "_")

def model_device(model):
    return next(model.parameters()).device

def load_causal_model_and_tokenizer_for_training(model_name_or_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_dir)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name_or_dir)
    model.config.pad_token_id = tokenizer.pad_token_id
    return model, tokenizer

def load_causal_model_and_tokenizer_for_eval(model_name_or_dir, eval_device=device):
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_dir)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name_or_dir)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(eval_device)
    model.eval()
    return model, tokenizer

def load_reward_model_and_tokenizer(model_dir, eval_device=device):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(eval_device)
    model.eval()
    return model, tokenizer

In [5]:
# Text generation and reward scoring helpers.
# generate_from_model() produces a response from an SFT or DPO model.
# score_with_reward_model() uses the trained reward model to assign a preference score.
def generate_from_model(model, tokenizer, prompt, max_new_tokens=80, do_sample=False):
    input_text = f"### Prompt:\n{prompt}\n\n### Response:\n"
    current_device = model_device(model)

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_GEN_INPUT_LENGTH,
    )
    inputs = {k: v.to(current_device) for k, v in inputs.items()}

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        gen_kwargs.update({"top_k": 50, "top_p": 0.95, "temperature": 0.8})

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Response:")[-1].strip()

def score_with_reward_model(prompt, response_text, reward_model, reward_tokenizer):
    rm_text = f"Prompt: {prompt}\nResponse: {response_text}"
    current_device = model_device(reward_model)

    inputs = reward_tokenizer(
        rm_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_RM_LENGTH,
    )
    inputs = {k: v.to(current_device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = reward_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)

    return probs[0][1].item()

In [6]:
# Preference-accuracy helpers.
# These functions compute the average token log-probability of a completion
# and use that to estimate whether a model prefers the chosen response over the rejected one.
def average_completion_logprob(model, tokenizer, prompt, completion, max_length=MAX_CAUSAL_LENGTH):
    current_device = model_device(model)

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]

    if len(completion_ids) == 0:
        return float("-inf")

    max_prompt_len = max_length - len(completion_ids)
    if max_prompt_len <= 0:
        completion_ids = completion_ids[: max_length - 1]
        prompt_ids = []
    else:
        prompt_ids = prompt_ids[-max_prompt_len:]

    input_ids = prompt_ids + completion_ids

    if len(input_ids) < 2:
        return float("-inf")

    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(current_device)

    with torch.no_grad():
        outputs = model(input_ids=input_tensor)
        logits = outputs.logits[:, :-1, :]
        labels = input_tensor[:, 1:]

    token_log_probs = torch.log_softmax(logits, dim=-1)
    gathered = torch.gather(token_log_probs, 2, labels.unsqueeze(-1)).squeeze(-1)

    start_idx = max(len(prompt_ids) - 1, 0)
    selected = gathered[:, start_idx:]

    if selected.numel() == 0:
        return float("-inf")

    return selected.mean().item()

def heldout_preference_accuracy(model_dir, pref_dataset, max_rows=100):
    model, tokenizer = load_causal_model_and_tokenizer_for_eval(model_dir, eval_device=device)
    rows = pref_dataset.select(range(min(max_rows, len(pref_dataset))))
    correct = 0

    for row in rows:
        chosen_lp = average_completion_logprob(model, tokenizer, row["prompt"], row["chosen"])
        rejected_lp = average_completion_logprob(model, tokenizer, row["prompt"], row["rejected"])
        if chosen_lp > rejected_lp:
            correct += 1

    return correct / len(rows)

bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

def compute_text_metrics(preds, refs):
    bleu = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])["bleu"]
    rouge = rouge_metric.compute(predictions=preds, references=refs)
    return {
        "bleu": bleu,
        "rouge1": rouge["rouge1"],
        "rouge2": rouge["rouge2"],
        "rougeL": rouge["rougeL"],
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## SFT dataset

In [7]:
# Small supervised fine-tuning (SFT) dataset.
# This custom instruction dataset is used to create the initial baseline instruction-following model.
# eval_df is a tiny evaluation set with reference answers for BLEU/ROUGE comparison.
sft_data = [
    {"prompt": "Explain reinforcement learning in simple terms.", "response": "Reinforcement learning is a type of machine learning where an agent learns by taking actions and receiving rewards or penalties."},
    {"prompt": "What is supervised learning?", "response": "Supervised learning is a method where a model learns from labeled examples that map inputs to known outputs."},
    {"prompt": "What is overfitting?", "response": "Overfitting happens when a model learns the training data too closely and performs poorly on new unseen data."},
    {"prompt": "Explain neural networks briefly.", "response": "Neural networks are models inspired by the brain that learn patterns using layers of connected computational units called neurons."},
    {"prompt": "What is a reward model in RLHF?", "response": "A reward model predicts which responses humans prefer so the language model can optimize toward preferred answers."},
    {"prompt": "Why is evaluation important in machine learning?", "response": "Evaluation helps measure how well a model performs and whether it generalizes to new data."},
    {"prompt": "What is the difference between AI and machine learning?", "response": "AI is the broader field of intelligent systems, while machine learning is a method that allows systems to learn from data."},
    {"prompt": "What is PPO in RLHF?", "response": "PPO is a reinforcement learning algorithm often used in RLHF to update the language model using reward signals."},
]

sft_df = pd.DataFrame(sft_data)
sft_df["text"] = sft_df.apply(
    lambda x: f"### Prompt:\n{x['prompt']}\n\n### Response:\n{x['response']}",
    axis=1,
)
sft_dataset = Dataset.from_pandas(sft_df[["text"]], preserve_index=False)

eval_df = pd.DataFrame([
    {"prompt": "Explain reinforcement learning.", "reference": "Reinforcement learning is a machine learning method where an agent learns by interacting with an environment and receiving rewards or penalties."},
    {"prompt": "What is overfitting?", "reference": "Overfitting happens when a model performs well on training data but poorly on unseen data."},
    {"prompt": "What is supervised learning?", "reference": "Supervised learning trains a model using labeled input-output examples."},
    {"prompt": "Why are reward models useful?", "reference": "Reward models estimate which responses humans prefer so the policy can improve toward better outputs."},
])

print(sft_dataset)
eval_df

Dataset({
    features: ['text'],
    num_rows: 8
})


,prompt,reference
0,Explain reinforcement learning.,Reinforcement learning is a machine learning m...
1,What is overfitting?,Overfitting happens when a model performs well...
2,What is supervised learning?,Supervised learning trains a model using label...
3,Why are reward models useful?,Reward models estimate which responses humans ...


## Baseline SFT

In [8]:
# SFT training function.
# For each base model, this trains on the small instruction dataset and saves the resulting baseline model.
def train_sft_model(base_model_name, train_dataset, output_dir, epochs=SFT_EPOCHS):
    model, tokenizer = load_causal_model_and_tokenizer_for_training(base_model_name)
    model.to(device)

    args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        num_train_epochs=epochs,
        logging_steps=1,
        save_strategy="epoch",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        processing_class=tokenizer,
    )
    trainer.train()
    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    return output_dir

In [9]:
# Train or reload the baseline SFT models, then evaluate them on the small reference set.
# If a saved model directory already exists, the code skips retraining to save time.
baseline_runs = []

for base_model_name in BASE_MODELS:
    run_dir = os.path.join(OUTPUT_ROOT, "sft_" + slugify(base_model_name))
    if not os.path.exists(run_dir):
        print(f"Training baseline SFT for {base_model_name}")
        train_sft_model(base_model_name, sft_dataset, run_dir)
    else:
        print(f"Skipping SFT for {base_model_name}; found existing directory.")

    model, tokenizer = load_causal_model_and_tokenizer_for_eval(run_dir, eval_device=device)
    preds = [generate_from_model(model, tokenizer, p, do_sample=False) for p in eval_df["prompt"]]
    metrics = compute_text_metrics(preds, eval_df["reference"].tolist())

    baseline_runs.append({
        "base_model": base_model_name,
        "baseline_model_dir": run_dir,
        **metrics
    })

baseline_results_df = pd.DataFrame(baseline_runs)
baseline_results_df

Training baseline SFT for distilgpt2


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (35 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

,base_model,baseline_model_dir,bleu,rouge1,rouge2,rougeL
0,distilgpt2,./assignment_outputs/sft_distilgpt2,0.00000,0.113072,0.031840,0.108122
1,gpt2,./assignment_outputs/sft_gpt2,0.06896,0.306205,0.102713,0.276768


## Preference datasets

In [10]:
# Load and preprocess the Anthropic HH-RLHF dataset.
# The original records contain full conversations, so we split them into:
#   prompt + chosen completion + rejected completion
# This gives us explicit preference pairs for reward modeling and DPO.
ASSISTANT_MARKER = "\n\nAssistant:"

def split_hh_prompt_completion(full_text: str):
    idx = full_text.rfind(ASSISTANT_MARKER)
    if idx == -1:
        return None, None
    prompt = full_text[: idx + len(ASSISTANT_MARKER)]
    completion = full_text[idx + len(ASSISTANT_MARKER):].strip()
    return prompt, completion

def hh_row_to_explicit_pref(row):
    prompt_c, chosen = split_hh_prompt_completion(row["chosen"])
    prompt_r, rejected = split_hh_prompt_completion(row["rejected"])
    if prompt_c is None or prompt_r is None:
        return None
    prompt = prompt_c if prompt_c == prompt_r else prompt_c
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

print("Loading Anthropic HH-RLHF...")
hh_raw = load_dataset("Anthropic/hh-rlhf")

hh_train_pref = []
for row in hh_raw["train"].shuffle(seed=SEED).select(range(HH_RLHF_TRAIN_PAIRS)):
    item = hh_row_to_explicit_pref(row)
    if item is not None:
        hh_train_pref.append(item)

hh_eval_pref = []
for row in hh_raw["test"].shuffle(seed=SEED).select(range(HH_RLHF_HELDOUT_PAIRS)):
    item = hh_row_to_explicit_pref(row)
    if item is not None:
        hh_eval_pref.append(item)

hh_train_pref = Dataset.from_pandas(pd.DataFrame(hh_train_pref), preserve_index=False)
hh_eval_pref = Dataset.from_pandas(pd.DataFrame(hh_eval_pref), preserve_index=False)

print(hh_train_pref)
print(hh_eval_pref)

Loading Anthropic HH-RLHF...


README.md: 0.00B [00:00, ?B/s]

harmless-base/train.jsonl.gz:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

helpful-base/train.jsonl.gz:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

helpful-online/train.jsonl.gz:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

helpful-rejection-sampled/train.jsonl.gz:   0%|          | 0.00/25.7M [00:00<?, ?B/s]

... (3 outputs left out)

helpful-rejection-sampled/test.jsonl.gz:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/160800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8552 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 4000
})
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 300
})


In [11]:
# Load and preprocess HelpSteer2.
# HelpSteer2 is not already in pairwise chosen/rejected form, so we group responses by prompt,
# score them using the provided annotation dimensions, and construct preference pairs by taking
# the highest-scoring response as 'chosen' and the lowest-scoring response as 'rejected'.
print("Loading HelpSteer2...")
hs_raw = load_dataset("nvidia/HelpSteer2")

def build_helpsteer_preferences(ds, max_pairs):
    grouped = defaultdict(list)

    for row in ds:
        prompt = row["prompt"]
        response = row["response"]
        score = (
            float(row.get("helpfulness", 0))
            + float(row.get("correctness", 0))
            + float(row.get("coherence", 0))
            + float(row.get("complexity", 0))
            + float(row.get("verbosity", 0))
        )
        grouped[prompt].append((response, score))

    pref_rows = []
    for prompt, items in grouped.items():
        if len(items) < 2:
            continue
        items = sorted(items, key=lambda x: x[1], reverse=True)
        best_response, best_score = items[0]
        worst_response, worst_score = items[-1]
        if best_score > worst_score:
            pref_rows.append({
                "prompt": prompt,
                "chosen": best_response,
                "rejected": worst_response
            })
        if len(pref_rows) >= max_pairs:
            break

    return Dataset.from_pandas(pd.DataFrame(pref_rows), preserve_index=False)

hs_train_pref = build_helpsteer_preferences(hs_raw["train"].shuffle(seed=SEED), HS_TRAIN_PAIRS)

if "validation" in hs_raw:
    hs_eval_pref = build_helpsteer_preferences(hs_raw["validation"].shuffle(seed=SEED), HS_HELDOUT_PAIRS)
else:
    hs_eval_pref = build_helpsteer_preferences(hs_raw["train"].shuffle(seed=SEED + 1), HS_HELDOUT_PAIRS)

print(hs_train_pref)
print(hs_eval_pref)

dpo_eval_ds = hh_eval_pref

Loading HelpSteer2...


README.md: 0.00B [00:00, ?B/s]

train.jsonl.gz:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.jsonl.gz:   0%|          | 0.00/582k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20324 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1038 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 4000
})
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 300
})


## Reward modeling

In [12]:
# Reward-model training helpers.
# We convert each pairwise preference example into two binary examples:
#   chosen -> label 1
#   rejected -> label 0
# Then we train a DistilBERT classifier to predict which responses are preferred.
def preference_to_unpaired_binary(pref_ds):
    rows = []
    for row in pref_ds:
        rows.append({"text": f"Prompt: {row['prompt']}\nResponse: {row['chosen']}", "label": 1})
        rows.append({"text": f"Prompt: {row['prompt']}\nResponse: {row['rejected']}", "label": 0})
    return Dataset.from_pandas(pd.DataFrame(rows), preserve_index=False)

def tokenize_reward_dataset(ds, rm_tokenizer):
    def tok(batch):
        return rm_tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_RM_LENGTH,
        )
    ds = ds.map(tok, batched=True)
    keep = {"input_ids", "attention_mask", "label"}
    for split_name in ds.keys():
        cols = ds[split_name].column_names
        ds[split_name] = ds[split_name].remove_columns([c for c in cols if c not in keep])
    return ds

accuracy_metric = evaluate.load("accuracy")

def compute_reward_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

def train_reward_model(run_name, pref_train_ds, output_dir):
    rm_train_unpaired = preference_to_unpaired_binary(pref_train_ds)
    rm_train_unpaired = rm_train_unpaired.train_test_split(test_size=0.1, seed=SEED)

    rm_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    tokenized = tokenize_reward_dataset(rm_train_unpaired, rm_tokenizer)

    rm_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    )
    rm_model.to(device)

    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        num_train_epochs=REWARD_EPOCHS,
        report_to="none",
        load_best_model_at_end=True,
    )

    trainer = Trainer(
        model=rm_model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["test"],
        compute_metrics=compute_reward_metrics,
    )
    trainer.train()

    trainer.save_model(output_dir)
    rm_tokenizer.save_pretrained(output_dir)
    return output_dir

In [13]:
# Train one reward model per preference dataset and save the directories for later reuse.
# This allows the notebook to compare multiple reward-data sources without retraining every time.
reward_model_runs = [
    ("hh_rlhf", hh_train_pref, hh_eval_pref),
    ("helpsteer2", hs_train_pref, hs_eval_pref),
]

reward_model_dirs = {}

for run_name, train_pref_ds, eval_pref_ds in reward_model_runs:
    out_dir = os.path.join(OUTPUT_ROOT, "reward_" + run_name)
    if not os.path.exists(out_dir):
        print(f"Training reward model for {run_name}")
        train_reward_model(run_name, train_pref_ds, out_dir)
    else:
        print(f"Skipping reward model {run_name}; found existing directory.")
    reward_model_dirs[run_name] = out_dir

reward_model_dirs

Training reward model for hh_rlhf


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

... (14 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'hh_rlhf': './assignment_outputs/reward_hh_rlhf',
 'helpsteer2': './assignment_outputs/reward_helpsteer2'}

## DPO training

In [14]:
# DPO training function.
# DPO (Direct Preference Optimization) fine-tunes the baseline SFT model directly from preference pairs.
# This is the main alignment step in the notebook.
def train_dpo_model(base_model_name, sft_model_dir, pref_train_ds, output_dir, epochs=DPO_EPOCHS):
    model, tokenizer = load_causal_model_and_tokenizer_for_training(sft_model_dir)
    model.to(device)

    if HAS_DPOCONFIG:
        dpo_args = DPOConfig(
            output_dir=output_dir,
            per_device_train_batch_size=1,
            num_train_epochs=epochs,
            logging_steps=10,
            save_strategy="epoch",
            report_to="none",
        )
    else:
        dpo_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=1,
            num_train_epochs=epochs,
            logging_steps=10,
            save_strategy="epoch",
            report_to="none",
        )

    try:
        trainer = DPOTrainer(
            model=model,
            args=dpo_args,
            train_dataset=pref_train_ds,
            processing_class=tokenizer,
        )
    except TypeError:
        trainer = DPOTrainer(
            model=model,
            args=dpo_args,
            train_dataset=pref_train_ds,
            tokenizer=tokenizer,
        )

    trainer.train()
    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    return output_dir

In [15]:
# Run DPO for each base model.
# Here we use HH-RLHF as the training preference dataset for the final alignment step.
# If model outputs already exist on disk, retraining is skipped.
dpo_runs_summary = []
dpo_train_ds = hh_train_pref

for base_model_name in BASE_MODELS:
    sft_dir = os.path.join(OUTPUT_ROOT, "sft_" + slugify(base_model_name))
    dpo_dir = os.path.join(OUTPUT_ROOT, "dpo_" + slugify(base_model_name))

    if not os.path.exists(dpo_dir):
        print(f"Training DPO model for {base_model_name}")
        train_dpo_model(base_model_name, sft_dir, dpo_train_ds, dpo_dir)
    else:
        print(f"Skipping DPO for {base_model_name}; found existing directory.")

    dpo_runs_summary.append({
        "base_model": base_model_name,
        "sft_dir": sft_dir,
        "dpo_dir": dpo_dir,
    })

dpo_runs_df = pd.DataFrame(dpo_runs_summary)
dpo_runs_df

Training DPO model for distilgpt2


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1648 > 1024). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

... (9 outputs left out)

Step,Training Loss
10,0.689387
20,0.712684
30,0.701777
40,0.719535
50,0.694899
60,0.741363
70,0.768127
80,0.705538
90,0.701709
100,0.736043


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,base_model,sft_dir,dpo_dir
0,distilgpt2,./assignment_outputs/sft_distilgpt2,./assignment_outputs/dpo_distilgpt2
1,gpt2,./assignment_outputs/sft_gpt2,./assignment_outputs/dpo_gpt2


## Held-out preference accuracy

In [16]:
# Evaluate held-out preference accuracy.
# This checks whether the baseline and DPO models assign higher likelihood to the chosen response
# than to the rejected response on a held-out preference dataset.
pref_eval_rows = []
for base_model_name in BASE_MODELS:
    sft_dir = os.path.join(OUTPUT_ROOT, "sft_" + slugify(base_model_name))
    dpo_dir = os.path.join(OUTPUT_ROOT, "dpo_" + slugify(base_model_name))

    pref_eval_rows.append({
        "base_model": base_model_name,
        "baseline_pref_accuracy": heldout_preference_accuracy(sft_dir, dpo_eval_ds, max_rows=100),
        "dpo_pref_accuracy": heldout_preference_accuracy(dpo_dir, dpo_eval_ds, max_rows=100),
    })

pref_eval_df = pd.DataFrame(pref_eval_rows)
pref_eval_df

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

,base_model,baseline_pref_accuracy,dpo_pref_accuracy
0,distilgpt2,0.54,0.57
1,gpt2,0.48,0.46


## Qualitative comparison

In [17]:
# Qualitative comparison.
# For each prompt, generate a response from the baseline SFT model and the DPO model,
# then score both with the HH-RLHF reward model for side-by-side comparison.
hh_reward_model, hh_reward_tokenizer = load_reward_model_and_tokenizer(
    reward_model_dirs["hh_rlhf"],
    eval_device=device
)

qualitative_rows = []
for base_model_name in BASE_MODELS:
    baseline_dir = os.path.join(OUTPUT_ROOT, "sft_" + slugify(base_model_name))
    dpo_dir = os.path.join(OUTPUT_ROOT, "dpo_" + slugify(base_model_name))

    baseline_model, baseline_tokenizer = load_causal_model_and_tokenizer_for_eval(
        baseline_dir, eval_device=device
    )
    dpo_model, dpo_tokenizer = load_causal_model_and_tokenizer_for_eval(
        dpo_dir, eval_device=device
    )

    for prompt in eval_df["prompt"]:
        baseline_response = generate_from_model(baseline_model, baseline_tokenizer, prompt, do_sample=False)
        dpo_response = generate_from_model(dpo_model, dpo_tokenizer, prompt, do_sample=False)

        qualitative_rows.append({
            "base_model": base_model_name,
            "prompt": prompt,
            "baseline_response": baseline_response,
            "baseline_reward_score": score_with_reward_model(
                prompt, baseline_response, hh_reward_model, hh_reward_tokenizer
            ),
            "dpo_response": dpo_response,
            "dpo_reward_score": score_with_reward_model(
                prompt, dpo_response, hh_reward_model, hh_reward_tokenizer
            ),
        })

qualitative_df = pd.DataFrame(qualitative_rows)
qualitative_df.head()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

,base_model,prompt,baseline_response,baseline_reward_score,dpo_response,dpo_reward_score
0,distilgpt2,Explain reinforcement learning.,The reinforcement learning model is a model th...,0.496631,Recognition is a method that learns from data ...,0.495440
1,distilgpt2,What is overfitting?,Overfitting is a method that uses a method to ...,0.493745,Overfitting is a technique that allows a model...,0.497681
2,distilgpt2,What is supervised learning?,Learning is a method that learns from data tha...,0.494958,Learning is a method that learns from data tha...,0.494958
3,distilgpt2,Why are reward models useful?,Reward models are a way to learn from reward m...,0.493624,Reward models are a way to reward systems that...,0.495684
4,gpt2,Explain reinforcement learning.,Reinforcement learning is a method that learns...,0.494476,Reinforcement learning is a method that learns...,0.494536


## Save CSV outputs

In [19]:
# Save the main outputs.
# These CSV files are useful for the report and for a small blind human evaluation table.
human_eval_template = qualitative_df[["base_model", "prompt", "baseline_response", "dpo_response"]].copy()
human_eval_template["preferred_system"] = ""
human_eval_path = os.path.join(OUTPUT_ROOT, "blind_human_eval_template.csv")
human_eval_template.to_csv(human_eval_path, index=False)

baseline_results_df.to_csv(os.path.join(OUTPUT_ROOT, "baseline_results.csv"), index=False)
pref_eval_df.to_csv(os.path.join(OUTPUT_ROOT, "heldout_preference_accuracy.csv"), index=False)
qualitative_df.to_csv(os.path.join(OUTPUT_ROOT, "qualitative_comparison.csv"), index=False)

print(os.path.join(OUTPUT_ROOT, "baseline_results.csv"))
print(os.path.join(OUTPUT_ROOT, "heldout_preference_accuracy.csv"))
print(os.path.join(OUTPUT_ROOT, "qualitative_comparison.csv"))
print(human_eval_path)

./assignment_outputs/baseline_results.csv
./assignment_outputs/heldout_preference_accuracy.csv
./assignment_outputs/qualitative_comparison.csv
./assignment_outputs/blind_human_eval_template.csv


## Notes for the marker

- The notebook is written to **skip retraining** when saved model directories already exist.
- The **baseline** is created with supervised fine-tuning on a small custom instruction dataset.
- **Reward models** are trained on public preference datasets.
- **DPO** is used as the preference-optimization / alignment step.
- Final results are reported through BLEU/ROUGE, held-out preference accuracy, reward scores, and qualitative examples.